<a href="https://colab.research.google.com/github/Marcin19721205/WSBNeuronowe/blob/main/atomski_14122025103929_R%C3%B3wnowa%C5%BCenie_klas_14_12_25.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Wprowadzenie

Kolejnym ważnym aspektem niemal każdego eksperymentu uczenia maszynowego jest możliwość równoważenia klas w przypadku, gdy ich proporcje są zaburzone. W większości rzeczywistych zadań, część klas może występować w niewielkiej ilości. Niestety, najczęściej będą to te najbardziej wartościowe i istotne. Istnieją narzędzia, które pozwalają radzić sobie z takimi zjawiskami.

W tym notebooku zapoznamy się z podstawowymi technikami, które mogą nam pomóc przywrócić właściwe proporcje w danych.

In [1]:
%pip install imblearn

In [5]:
%pip install tensorflow_addons

ERROR: Could not find a version that satisfies the requirement tensorflow_addons (from versions: none)
ERROR: No matching distribution found for tensorflow_addons


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
#import tensorflow_addons as tfa
import gc
import tensorflow.keras as krs

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, accuracy_score, precision_score, recall_score

In [8]:
%matplotlib inline

# Wczytanie danych



W ramach tego ćwiczenia będziemy pracować na zbiorze danych do klasyfikacji wieloklasowej, gdzie porporcje pomiędzy klasami są dość silnie zaburzone.

<div class='alert alert-block alert-warning'>
    Wczytaj zbiór danych o nazwie <b>imbalanced_dataset.csv</b>. Sprawdź proporcje pomiędzy klasami, zawartymi w kolumnie <b>y</b>.
</div>

Dane są dostępne pod adresem `https://drive.google.com/uc?id=1u2M_HRvD_MXJ7Kztbyr7i9FvvW0Ms6TC&export=download`

In [3]:
data = pd.read_csv("https://drive.google.com/uc?id=1u2M_HRvD_MXJ7Kztbyr7i9FvvW0Ms6TC&export=download")

In [4]:
data['y'].value_counts()

,count
y,
0,3490
1,1006
2,504


<div class='alert alert-block alert-warning'>
    <b>Zadanie</b>:
    <ol>
        <li>oddziel kolumnę y od całej reszty danych. Zapisz ją pod zmienną y</li>
        <li>dane, które pozostają - zapisz pod zmienną X</li>
        <li>zrób rzutowanie typów zmiennej X na typ <code>np.float32</code> ze względu na kompatybilność z tensorflow</li>
        <li>zamień klasy wektora <code>y</code> na postać one-hot i zapisz ponownie pod zmienną y</li>
    </ol>
</div>

In [5]:
X = data.drop('y', axis=1).astype(np.float32)
y = data['y']

In [6]:
y = tf.keras.utils.to_categorical(y)

Sprawdzenie poprawności wyników:

In [7]:
assert X.shape == (5000, 20)
assert (X.dtypes == np.float32).all()

assert y.shape == (5000, 3)

<div class='alert alert-block alert-warning'>
    Podziel dane na train i test w proporcji <code>train = 0.8% zbioru, random_state = 123</code>
</div>

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=123)

Sprawdzenie poprawności wyniku

In [9]:
assert X_train.shape == (4000, 20)
assert y_train.shape == (4000, 3)

assert X_test.shape == (1000, 20)
assert y_test.shape == (1000, 3)

assert X_train.values.dtype == np.float32
assert X_test.values.dtype == np.float32

# Bazowe modele

Zaczniemy od zbudowania bazowego modelu, który będzie operował na oryginalnych danych, bez równoważenia. Stwrzomy dwa modele:

1. Zawsze przewidujący najczęstszą klasę
2. Prostą sieć neuronową do klasyfikacji wieloklasowej

Sieć neuronową o zadanej architekturze będziemy szkolić od zera kilkukrotnie, odpowiednio manipulując wcześniej danymi.

<div class='alert alert-block alert-danger'>
    <b>UWAGA</b> w tym ćwiczeniu nie skupiamy się na stworzeniu jak najlepszej architektury sieci neurnowej dla zadanego problemu. Chcemy za to zbadać wpływ równowagi klas lub jej braku na jakość predykcji. Nie skupiaj się więc na aspekcie doboru jak najlepszej sieci, ale na operacjach na danych, które za chwilę będziemy wykonywać.
</div>

## Prosty klasyfikator

<div class='alert alert-block alert-warning'>
    Zbuduj klasyfikator, zawsze przewidujący najczęściej występującą klasę. Wykorzystaj implementajcę <code>sklearn.dummy.DummyClassifier</code>
</div>


In [10]:
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(X_train, y_train)

DummyClassifier(strategy='most_frequent')

Sprawdzenie poprawności wyniku

In [11]:
yhat_dummy = dummy.predict(X_test)
assert np.round(accuracy_score(y_test, yhat_dummy), 3) == 0.694
print(classification_report(y_test, yhat_dummy, zero_division=0))

clf_rep = classification_report(y_test, yhat_dummy, zero_division=0, output_dict=True)
assert np.round(clf_rep['0']['precision'], 3) == 0.694
assert clf_rep['1']['precision'] == 0.0

assert clf_rep['0']['recall'] == 1.0
assert clf_rep['1']['recall'] == 0.0

              precision    recall  f1-score   support

           0       0.69      1.00      0.82       694
           1       0.00      0.00      0.00       215
           2       0.00      0.00      0.00        91

   micro avg       0.69      0.69      0.69      1000
   macro avg       0.23      0.33      0.27      1000
weighted avg       0.48      0.69      0.57      1000
 samples avg       0.69      0.69      0.69      1000



<div class='alert alert-block alert-info'>
    W powyższych wynikach widać trzy niepokojące rzeczy:
    
<ol>
<li>Głupi klasyfikator potrafi "osiągnąć" trafnośc na poziomie 69%</li>
<li>Metryki takie jak trafność są bezużyteczne w przypadku braku zrównoważenia klas</li>
<li>Dopiero łączne wykorzystanie metryk precyzji, czułości oraz F1 pozwala zobaczyć skalę problemu</li>
</ol>
</div>

## Sieć neuronowa

<div class='alert alert-block alert-warning'>
    Przygotuj funkję, która buduje i zwraca gotową sieć neuronową o następującej specyfikacji:

<ol>
<li>Warstwy: <code>BatchNorm - Dense(32, relu) - Dense(16, relu) - Dense(3, softmax)</code></li>
<li>Dodatkowe opcje: <code>optymalizator=Adam, koszt=categorical_crossentropy, metryki: accuracy, F1Score(num_classes=3, average=macro)</code></li>
</ol>
</div>

<div class='alert alert-block alert-info'>
    Metryka F1Score zawarta są w bardzo przydatnym pakiecie <code>tensorflow_addons</code>. Warto przeczytać dokumentację tego narzędzia.
</div>


In [18]:
import tensorflow as tf
import numpy as np

# Custom F1-score metric for Keras, replacing tfa.metrics.F1Score due to installation issues.
# This metric calculates the macro-averaged F1-score across all classes.
class F1Macro(tf.keras.metrics.Metric):
    def __init__(self, name='f1_score', num_classes=3, **kwargs):
        super(F1Macro, self).__init__(name=name, **kwargs)
        self.num_classes = num_classes
        self.true_positives = self.add_weight(name='tp', initializer='zeros', shape=(num_classes,))
        self.false_positives = self.add_weight(name='fp', initializer='zeros', shape=(num_classes,))
        self.false_negatives = self.add_weight(name='fn', initializer='zeros', shape=(num_classes,))

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true_labels = tf.argmax(y_true, axis=-1)  # (batch_size,)
        y_pred_labels = tf.argmax(y_pred, axis=-1)  # (batch_size,)

        # Convert labels to one-hot for easier comparison in a vectorized manner
        y_true_one_hot = tf.one_hot(y_true_labels, self.num_classes) # (batch_size, num_classes)
        y_pred_one_hot = tf.one_hot(y_pred_labels, self.num_classes) # (batch_size, num_classes)

        # Calculate true positives (TP) for all classes
        tp = tf.reduce_sum(y_true_one_hot * y_pred_one_hot, axis=0) # (num_classes,)

        # Calculate false positives (FP) for all classes
        # FP = (predicted_positive) AND (actual_negative)
        fp = tf.reduce_sum((1 - y_true_one_hot) * y_pred_one_hot, axis=0) # (num_classes,)

        # Calculate false negatives (FN) for all classes
        # FN = (actual_positive) AND (predicted_negative)
        fn = tf.reduce_sum(y_true_one_hot * (1 - y_pred_one_hot), axis=0) # (num_classes,)

        self.true_positives.assign_add(tp)
        self.false_positives.assign_add(fp)
        self.false_negatives.assign_add(fn)

    def result(self):
        # Calculate precision per class
        precision = self.true_positives / (self.true_positives + self.false_positives + tf.keras.backend.epsilon())

        # Calculate recall per class
        recall = self.true_positives / (self.true_positives + self.false_negatives + tf.keras.backend.epsilon())

        # Calculate F1-score per class
        f1_scores = 2 * (precision * recall) / (precision + recall + tf.keras.backend.epsilon())

        # Replace NaN with 0.0 (e.g., when precision and recall are both 0)
        f1_scores = tf.where(tf.math.is_nan(f1_scores), 0.0, f1_scores)

        return tf.reduce_mean(f1_scores) # macro average

    def reset_state(self):
        self.true_positives.assign(tf.zeros(self.num_classes))
        self.false_positives.assign(tf.zeros(self.num_classes))
        self.false_negatives.assign(tf.zeros(self.num_classes))

def build_model():
    model = krs.Sequential([
        krs.layers.BatchNormalization(input_shape=(X_train.shape[1],)),
        krs.layers.Dense(32, activation='relu'),
        krs.layers.Dense(16, activation='relu'),
        krs.layers.Dense(3, activation='softmax')
    ])
    model.compile(optimizer='Adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy', F1Macro(num_classes=3)])
    return model

In [19]:
model1 = build_model()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/normalization/batch_normalization.py:142: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Sprawdzenie poprawności wyniku:

In [20]:
assert 'batch_normalization' in model1.layers[0].name

assert 'dense' in model1.layers[1].name
assert model1.layers[1].units == 32
assert model1.layers[1].activation is tf.keras.activations.relu

assert 'dense' in model1.layers[2].name
assert model1.layers[2].units == 16
assert model1.layers[2].activation is tf.keras.activations.relu

assert 'dense' in model1.layers[3].name
assert model1.layers[3].units == 3
assert model1.layers[3].activation is tf.keras.activations.softmax

<div class='alert alert-block alert-warning'>
    Wyszkol przygotowany model przez 5 epok (batch size 32) na zbiorze treningowym. Przeprowadź ewaluację na zbiorze testowym i zapisz wynik. Przy ewaluacji wybierz opcję <code>return_dict=True</code>. W ten sposób, o szkoleniu otrzymamy słownik z wartościami poszczególnych metryk. Zapisz go pod zmienną m1_eval.
</div>

In [23]:
model1.fit(X_train, y_train, epochs=5, batch_size=32)
m1_eval = model1.evaluate(X_test, y_test, return_dict=True)

Epoch 1/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8223 - f1_score: 0.6657 - loss: 0.4628
Epoch 2/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8326 - f1_score: 0.6931 - loss: 0.4404
Epoch 3/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8095 - f1_score: 0.6551 - loss: 0.4656
Epoch 4/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8273 - f1_score: 0.6800 - loss: 0.4399
Epoch 5/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8350 - f1_score: 0.6947 - loss: 0.4170
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.8444 - f1_score: 0.7019 - loss: 0.4314 


In [24]:
m1_eval

{'accuracy': 0.8420000076293945,
 'f1_score': 0.6850001811981201,
 'loss': 0.43366843461990356}

Sprawdzenie poprawności wyniku:

In [25]:
assert m1_eval['accuracy'] >= 0.7
assert 0.45 <=  m1_eval['f1_score']

<div class='alert alert-block alert-warning'>
    Dokonaj predykcji na zbiorze testowym. Wszystkie obiekty, które osiągną próg pewności >=0.5 zalicz do klasy 1. Wyświetl podsumowanie klasyfikacji <code>classification_report</code> z pakietu sklearn.
</div>

In [26]:
y_pred_proba = model1.predict(X_test)
yhat_model1 = np.argmax(y_pred_proba, axis=1)
y_test_labels = np.argmax(y_test, axis=1)

print(classification_report(y_test_labels, yhat_model1, zero_division=0))

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step
              precision    recall  f1-score   support

           0       0.86      0.97      0.91       694
           1       0.84      0.66      0.74       215
           2       0.56      0.32      0.41        91

    accuracy                           0.84      1000
   macro avg       0.75      0.65      0.69      1000
weighted avg       0.83      0.84      0.83      1000



<div class='alert alert-block alert-info'>
    Jak widać, czułośc (ang. *recall*) i precyzja  (ang. *precision*) dla klas o małej liczności nie są zbyt dobre. Spróbujemy je poprawić <b>równoważąc klasy w próbce uczącej.</b>
</div>

# Równoważenie klas

Poniżej zostaną zaprezentowane sposoby równoważenia klas, należące do kategorii opisywanych na wykładzie. Zastosujemy kilka z nich i sprawdzimy, czy dają oczekiwane rezultaty.

Zaczniemy od zaimportowania biblioteki, w której zawarte są odpowiednie narzędzia.

In [27]:
import imblearn

## Oversampling

Pierwszą z metod będzie 'dolosowywanie' obiektów z klasy mniejszościowej. Zrobimy to dwoma sposobami.

### Random

Pierwszy sposób dolosowtwania klasy mniejszościowej polega na losowym wyborze obiektów z klas mniejszościowych tak długo, aż poszczególne ilości się zrównoważą.

<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Z modułu <code>imblearn.over_sampling</code> zaimportuj obiekt <code>RandomOverSampler</code>. </li>
        <li>Utwórz obiekt klasy <code>RandomOverSampler</code> ustawiając random_state=999</li>
        <li>Wywołaj funkcję <code>.fit_transform(..., ...)</code> obiektu RandomOverSamplera, szkoląc go na treningowych danych X i y</li>
        <li>W procesie szkolenia oversampler przeliczy klasy i dokona ich równoważenia, zwracając nowy zbiór danych</li>
        <li>Zapisz docelowy zbiór danych pod zmiennymi <code>X_train_ros, y_train_ros</code></li>
        <li>Sprawdź proporcje klas w zbiorze treningowym - zapisz je w postaci <b>słowika (dict) pod zmienną ycnt_ros: [klucz: numer klasy]: [wartość: [%] w zbiorze treningowym]</b></li>
    </ol>
</div>

In [46]:
from imblearn.over_sampling import RandomOverSampler

y_train_labels = np.argmax(y_train, axis=1)

ros = RandomOverSampler(random_state=999)
X_train_ros, y_train_ros_labels = ros.fit_resample(X_train, y_train_labels)

y_train_ros = tf.keras.utils.to_categorical(y_train_ros_labels, num_classes=3)

In [47]:

ros_value_counts = pd.Series(y_train_ros_labels).value_counts(normalize=True)
ycnt_ros = {class_label: proportion for class_label, proportion in ros_value_counts.items()}

Sprawdzenie poprawności wyniku:

In [36]:
for i in range(3):
    assert round(ycnt_ros[i], 3) == 0.333

<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Wykorzystując napisaną wcześniej funkcję- utwórz nowy obiekt sieci neurnowej</li>
        <li>Wyszkol przygotowany model przez 5 epok (batch size 32) na zrównoważonym zbiorze treningowym. </li>
        <li>Przeprowadź ewaluację na zbiorze testowym i zapisz wynik. Przy ewaluacji wybierz opcję <code>return_dict=True</code>. W ten sposób, o szkoleniu otrzymamy słownik z wartościami poszczególnych metryk. Zapisz go pod zmienną m2_eval</li>
        <li>Wyświetl podsumowanie klasyfikacji <code>classification_report</code> z pakietu sklearn.</li>
    </ol>
</div>

In [37]:
model2 = build_model()
model2.fit(X_train_ros, y_train_ros, epochs=5, batch_size=32)
m2_eval = model2.evaluate(X_test, y_test, return_dict=True)

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/normalization/batch_normalization.py:142: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


263/263 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.4392 - f1_score: 0.4307 - loss: 1.0883
Epoch 2/5
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6613 - f1_score: 0.6602 - loss: 0.7734
Epoch 3/5
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7253 - f1_score: 0.7251 - loss: 0.6357
Epoch 4/5
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7642 - f1_score: 0.7638 - loss: 0.5792
Epoch 5/5
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7862 - f1_score: 0.7857 - loss: 0.5094
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7434 - f1_score: 0.6737 - loss: 0.5746


In [40]:
m2_eval

{'accuracy': 0.7400000095367432,
 'f1_score': 0.6668456196784973,
 'loss': 0.5921933650970459}

In [39]:
assert m2_eval['accuracy'] >= 0.7
assert 0.65 <=  m2_eval['f1_score']

In [38]:
y_pred_proba_m2 = model2.predict(X_test)
yhat_model2 = np.argmax(y_pred_proba_m2, axis=1)
y_test_labels = np.argmax(y_test, axis=1)

print(classification_report(y_test_labels, yhat_model2, zero_division=0))

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
              precision    recall  f1-score   support

           0       0.93      0.74      0.82       694
           1       0.68      0.69      0.69       215
           2       0.34      0.87      0.49        91

    accuracy                           0.74      1000
   macro avg       0.65      0.77      0.67      1000
weighted avg       0.82      0.74      0.76      1000



<div class='alert alert-block alert-info'>
    Wyniki powinny się zmienić w stosunku do scenariusza bazowego - najprawdopodobniej spadła dokładność (ang. <i>accuracy</i>) ale wzrosła czułość i precyzja dla co najmniej jednej klasy mniejszościowej (1 i 2). To jest spodziewany efekt. Będziemy szukać dalej, czy inne procedury równoważenia zapewnią lepsze wyniki.
</div>

### SMOTE

Durgą procedurą równoważenia próbek, którą wykorzystamy będzie SMOTE, omawiane na wykładach. Ta metoda tworzy syntetyczne próbki, powtałe na przecięciu odcinków łączących obiekty z klasy mniejszościowej.

<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Z modułu <code>imblearn.over_sampling</code> zaimportuj obiekt <code>SMOTE</code>. </li>
        <li>Utwórz obiekt klasy <code>SMOTE</code> ustawiając random_state=999</li>
        <li>Wywołaj funkcję <code>.fit_transform(..., ...)</code> obiektu SMOTE, szkoląc go na treningowych danych X i y</li>
        <li>W procesie szkolenia oversampler przeliczy klasy i dokona ich równoważenia, zwracając nowy zbiór danych</li>
        <li>Zapisz docelowy zbiór danych pod zmiennymi <code>X_train_smote, y_train_smote</code></li>
        <li>Sprawdź proporcje klas w zbiorze treningowym - zapisz je w postaci <b>słowika (dict) pod zmienną ycnt_smote: [klucz: numer klasy]: [wartość: [%] w zbiorze treningowym]</b></li>
    </ol>
</div>

In [41]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=999)
X_train_smote, y_train_smote_labels = smote.fit_resample(X_train, np.argmax(y_train, axis=1))
y_train_smote = tf.keras.utils.to_categorical(y_train_smote_labels, num_classes=3)

In [42]:
smote_value_counts = pd.Series(y_train_smote_labels).value_counts(normalize=True)
ycnt_smote = {class_label: proportion for class_label, proportion in smote_value_counts.items()}

Sprawdzenie poprawności wyniku

In [43]:
for i in range(3):
    assert round(ycnt_smote[i], 3) == 0.333

<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Wykorzystując napisaną wcześniej funkcję- utwórz nowy obiekt sieci neurnowej</li>
        <li>Wyszkol przygotowany model przez 5 epok (batch size 32) na zrównoważonym zbiorze treningowym. </li>
        <li>Przeprowadź ewaluację na zbiorze testowym i zapisz wynik. Przy ewaluacji wybierz opcję <code>return_dict=True</code>. W ten sposób, o szkoleniu otrzymamy słownik z wartościami poszczególnych metryk. Zapisz go pod zmienną m3_eval</li>
        <li>Wyświetl podsumowanie klasyfikacji <code>classification_report</code> z pakietu sklearn.</li>
    </ol>
</div>

In [44]:
model3 = build_model()
model3.fit(X_train_smote, y_train_smote, epochs=5, batch_size=32)
m3_eval = model3.evaluate(X_test, y_test, return_dict=True)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/normalization/batch_normalization.py:142: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5
263/263 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.4171 - f1_score: 0.3920 - loss: 1.0604
Epoch 2/5
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6540 - f1_score: 0.6527 - loss: 0.7685
Epoch 3/5
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7282 - f1_score: 0.7278 - loss: 0.6078
Epoch 4/5
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7746 - f1_score: 0.7738 - loss: 0.5299
Epoch 5/5
263/263 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8073 - f1_score: 0.8070 - loss: 0.4782
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7643 - f1_score: 0.6883 - loss: 0.5558


In [48]:
m3_eval

{'accuracy': 0.7770000100135803,
 'f1_score': 0.6972296237945557,
 'loss': 0.5464316010475159}

In [52]:
assert 0.7 <= m3_eval['accuracy']
assert 0.6 <= m3_eval['f1_score']

In [53]:
y_pred_proba_m3 = model3.predict(X_test)
yhat_model3 = np.argmax(y_pred_proba_m3, axis=1)
y_test_labels = np.argmax(y_test, axis=1)

print(classification_report(y_test_labels, yhat_model3, zero_division=0))

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
              precision    recall  f1-score   support

           0       0.94      0.78      0.85       694
           1       0.70      0.77      0.73       215
           2       0.38      0.78      0.51        91

    accuracy                           0.78      1000
   macro avg       0.67      0.78      0.70      1000
weighted avg       0.84      0.78      0.80      1000



<div class='alert alert-block alert-info'>
    Wyniki powinny się zmienić w stosunku do scenariusza bazowego - najprawdopodobniej spadła dokładność (ang. <i>accuracy</i>) ale wzrosła czułość i precyzja dla co najmniej jednej klasy mniejszościowej (1 i 2). To jest spodziewany efekt. Będziemy szukać dalej, czy inne procedury równoważenia zapewnią lepsze wyniki.<br>
    Porównaj otrzymane wyniki z RandomOverSampling'iem. Czy jest lepiej, czy gorzej? Jeśli tak, to w jakim zakresie (w odniesieniu do której klasy?).
</div>

## Under sampling

Kolejna grupa procedur to zmniejszenie liczności klasy większościowej - odrzucenie nadmiarowych obserwaci, aby zredukować ją do takiej samej liczności, jak klasa mniejszościowa. Ta metoda niestety powoduje odrzucenie znacznej ilości użytecznych danych - z tego powodu może być czasem problematyczna.

<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Z modułu <code>imblearn.over_sampling</code> zaimportuj obiekt <code>RandomUnderSampler</code>. </li>
        <li>Utwórz obiekt klasy <code>RandomUnderSampler</code> ustawiając random_state=999</li>
        <li>Wywołaj funkcję <code>.fit_transform(..., ...)</code> obiektu RandomUnderSampler, szkoląc go na treningowych danych X i y</li>
        <li>W procesie szkolenia oversampler przeliczy klasy i dokona ich równoważenia, zwracając nowy zbiór danych</li>
        <li>Zapisz docelowy zbiór danych pod zmiennymi <code>X_train_rus, y_train_rus</code></li>
        <li>Sprawdź proporcje klas w zbiorze treningowym - zapisz je w postaci <b>słowika (dict) pod zmienną ycnt_rus: [klucz: numer klasy]: [wartość: [%] w zbiorze treningowym]</b></li>
    </ol>
</div>

In [62]:
from imblearn.under_sampling import RandomUnderSampler


y_train_labels = np.argmax(y_train, axis=1)

rus = RandomUnderSampler(random_state=999)
X_train_rus, y_train_rus_labels = rus.fit_resample(X_train, y_train_labels)


y_train_rus = tf.keras.utils.to_categorical(y_train_rus_labels, num_classes=3)

In [63]:
rus_value_counts = pd.Series(y_train_rus_labels).value_counts(normalize=True)
ycnt_rus = {class_label: proportion for class_label, proportion in rus_value_counts.items()}

Sprawdzenie poprawności wyniku:

In [64]:
for i in range(3):
    assert round(ycnt_rus[i], 3) == 0.333

<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Wykorzystując napisaną wcześniej funkcję- utwórz nowy obiekt sieci neurnowej</li>
        <li>Wyszkol przygotowany model przez 5 epok (batch size 32) na zrównoważonym zbiorze treningowym. </li>
        <li>Przeprowadź ewaluację na zbiorze testowym i zapisz wynik. Przy ewaluacji wybierz opcję <code>return_dict=True</code>. W ten sposób, o szkoleniu otrzymamy słownik z wartościami poszczególnych metryk. Zapisz go pod zmienną m4_eval</li>
        <li>Wyświetl podsumowanie klasyfikacji <code>classification_report</code> z pakietu sklearn.</li>
    </ol>
</div>

In [65]:
model4 = build_model()
model4.fit(X_train_rus, y_train_rus, epochs=5, batch_size=32)
m4_eval = model4.evaluate(X_test, y_test, return_dict=True)

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/normalization/batch_normalization.py:142: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


39/39 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.3781 - f1_score: 0.3290 - loss: 1.1147
Epoch 2/5
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4374 - f1_score: 0.4214 - loss: 1.0539
Epoch 3/5
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5182 - f1_score: 0.5156 - loss: 0.9957
Epoch 4/5
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5648 - f1_score: 0.5636 - loss: 0.9413
Epoch 5/5
39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5722 - f1_score: 0.5712 - loss: 0.9166
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6011 - f1_score: 0.5028 - loss: 0.9463


In [66]:
m4_eval

{'accuracy': 0.5960000157356262,
 'f1_score': 0.49241891503334045,
 'loss': 0.9466121792793274}

In [67]:
assert 0.4 <= m4_eval['f1_score']
assert 0.4 <= m4_eval['accuracy']

In [68]:
y_pred_proba_m4 = model4.predict(X_test)
yhat_model4 = np.argmax(y_pred_proba_m4, axis=1)
y_test_labels = np.argmax(y_test, axis=1)

print(classification_report(y_test_labels, yhat_model4, zero_division=0))

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
              precision    recall  f1-score   support

           0       0.85      0.64      0.73       694
           1       0.47      0.50      0.48       215
           2       0.18      0.49      0.26        91

    accuracy                           0.60      1000
   macro avg       0.50      0.54      0.49      1000
weighted avg       0.71      0.60      0.64      1000



<div class='alert alert-block alert-info'>
   W tym przypadku wyniki powinny być znacznie gorsze, niż przy wykorzystaniu wcześniejszych podejść oraz w modelu bazowym. Wynika to z faktu, że RandomUnderSampler odrzuca obiekty (rekordy), które mogą nieść ze sobą bardzo użyteczne informacje.
    <br>
    <br>
    Nie znaczy to, że UnderSampling nie jest przydatny - najcześciej wykorzystuje się go w sytuacjach, gdy mamy bardzo dużo danych, które niekoniecznie muszą być użyteczne (np. macierze rzadkie w systemach rekomendacyjnych, etc.).
    <br>
    <br>
    W tym konkretnym przypadku - raczej nam się nie przyda.
</div>

## SMOTETomek - upsampling i undersampling jedncześnie

Jak łatwo się domyślić, opisane wyżej metody można połączyć, stosując jednocześnie syntetyczny oversampling oraz redukcję niektórych obserwacji z klasy większościowej. Spróbujmy i sprawdźmy, czy ta metoda da lepsze rezultaty niż np. wyłącznie SMOTE.

<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Z modułu <code>imblearn.over_sampling</code> zaimportuj obiekt <code>SMOTETomek</code>. </li>
        <li>Utwórz obiekt klasy <code>SMOTETomek</code> ustawiając random_state=999</li>
        <li>Wywołaj funkcję <code>.fit_transform(..., ...)</code> obiektu SMOTETomek, szkoląc go na treningowych danych X i y</li>
        <li>W procesie szkolenia oversampler przeliczy klasy i dokona ich równoważenia, zwracając nowy zbiór danych</li>
        <li>Zapisz docelowy zbiór danych pod zmiennymi <code>X_train_smotet, y_train_smotet</code></li>
        <li>Sprawdź proporcje klas w zbiorze treningowym - zapisz je w postaci <b>słowika (dict) pod zmienną ycnt_smotet: [klucz: numer klasy]: [wartość: [%] w zbiorze treningowym]</b></li>
    </ol>
</div>

In [81]:

from imblearn.combine import SMOTETomek
smotet = SMOTETomek(random_state=999)
X_train_smotet, y_train_smotet_labels = smotet.fit_resample(X_train, np.argmax(y_train, axis=1))
y_train_smotet = tf.keras.utils.to_categorical(y_train_smotet_labels, num_classes=3)

In [82]:
ycnt_smotet = {class_label: proportion for class_label, proportion in pd.Series(y_train_smotet_labels).value_counts(normalize=True).items()}

Sprawdzenie poprawności wyniku:

In [83]:
for i in range(3):
    assert round(ycnt_smotet[i], 3) == 0.333

<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Wykorzystując napisaną wcześniej funkcję- utwórz nowy obiekt sieci neurnowej</li>
        <li>Wyszkol przygotowany model przez 5 epok (batch size 32) na zrównoważonym zbiorze treningowym. </li>
        <li>Przeprowadź ewaluację na zbiorze testowym i zapisz wynik. Przy ewaluacji wybierz opcję <code>return_dict=True</code>. W ten sposób, o szkoleniu otrzymamy słownik z wartościami poszczególnych metryk. Zapisz go pod zmienną m5_eval</li>
        <li>Wyświetl podsumowanie klasyfikacji <code>classification_report</code> z pakietu sklearn.</li>
    </ol>
</div>

In [84]:
model5 = build_model()
model5.fit(X_train_smotet, y_train_smotet, epochs=5, batch_size=32)
m5_eval = model5.evaluate(X_test, y_test, return_dict=True)

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/normalization/batch_normalization.py:142: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


262/262 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.4226 - f1_score: 0.3977 - loss: 1.0853
Epoch 2/5
262/262 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6783 - f1_score: 0.6782 - loss: 0.7667
Epoch 3/5
262/262 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7504 - f1_score: 0.7500 - loss: 0.6115
Epoch 4/5
262/262 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7861 - f1_score: 0.7853 - loss: 0.5164
Epoch 5/5
262/262 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8188 - f1_score: 0.8182 - loss: 0.4612
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7963 - f1_score: 0.7067 - loss: 0.5053


In [85]:
m5_eval

{'accuracy': 0.8040000200271606,
 'f1_score': 0.7116227149963379,
 'loss': 0.5025611519813538}

In [86]:
assert 0.7 <= m5_eval['accuracy']
assert 0.65 <= m5_eval['f1_score']

In [87]:
y_pred_proba_m5 = model5.predict(X_test)
yhat_model5 = np.argmax(y_pred_proba_m5, axis=1)
y_test_labels = np.argmax(y_test, axis=1)

print(classification_report(y_test_labels, yhat_model5, zero_division=0))

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
              precision    recall  f1-score   support

           0       0.93      0.84      0.88       694
           1       0.81      0.69      0.75       215
           2       0.38      0.77      0.51        91

    accuracy                           0.80      1000
   macro avg       0.71      0.77      0.71      1000
weighted avg       0.85      0.80      0.82      1000



<div class='alert alert-block alert-info'>
 W zależności od przebiegu uczenia, wyniki będą zbliżone lub nieznacznie odbiegające od tych, któe daje SMOTE. Znacząco powinna wzrosnąć czułość (wykrywalność) klas mniejszościowych w stosunku do scenariusza bazowego, kosztem precyzji. Innymi słowy - model częściej znajduje obiekty klasy mniejszościowej, ale jednocześnie zaczyna się mylić robiąć takie przypisania.
</div>

# Nadawanie wag klasom

Jeszcze jednym sposobem na szkolenie sieci neuronowej do rozpoznawania obiektów klasy mniejszościowej, jest nadanie wag poszczególnym klasom. Działa to w sposób następujący:

1. Każdej klasie nadajemy jakąś wagę.
2. Podczas procesu uczenia się, dla obiektów danej klasy, funkcja kosztu (np. entropia krzyżowa, ang. *Cross entropy*) jest wymnażana przez tą wagę
3. W ten sposób, obiekty określonej klasy mogą ważyć więcej lub mniej w przypadku ich błędnej klasyfikacji i tym samym silniej lub słabiej wpływać na dopasowanie funkcji kosztu.


W naszym przykładzie spróbujemy **bez równoważenia zbioru** nadać klasie o najmniejszej liczności (2) wagę = 0.5, drugiej mniej licznej klasie (1) wagę 0.35 i klasie więszkościowej wagę 0.15, aby położyć większy nacisk na 1 i 2.

<div class='alert alert-block alert-warning'>
    <b>Zadanie:</b> utwórz słównik, określający wagi poszczgólnych klas w sposób następujący:
    <il>
        <li>Klasa 0: waga 0.15</li>
        <li>Klasa 1: waga 0.35</li>
        <li>Klasa 2: waga 0.5</li>
    </il>
</div>

In [77]:
class_weights = {0: 0.15, 1: 0.35, 2: 0.5}

<div class='alert alert-block alert-warning'>
    <b>Zadania:</b>
    <ol>
        <li>Wykorzystując napisaną wcześniej funkcję- utwórz nowy obiekt sieci neurnowej</li>
        <li>Wyszkol przygotowany model przez 5 epok (batch size 32) na <b>PODSTAWOWYM zbiorze treningowym</b> bez równoważenia </li>
        <li>Do funkcji <code>fit()</code> sieci neuronowej, przekaż dodatkowy argument <code>class_weights=</code>zawierający określone wcześniej wagi klas.<li>
        <li>Przeprowadź ewaluację na zbiorze testowym i zapisz wynik. Przy ewaluacji wybierz opcję <code>return_dict=True</code>. W ten sposób, o szkoleniu otrzymamy słownik z wartościami poszczególnych metryk. Zapisz go pod zmienną m6_eval</li>
        <li>Wyświetl podsumowanie klasyfikacji <code>classification_report</code> z pakietu sklearn.</li>
    </ol>
</div>

In [78]:
model6 = build_model()
model6.fit(X_train, y_train, epochs=5, batch_size=32, class_weight=class_weights)
m6_eval = model6.evaluate(X_test, y_test, return_dict=True)

Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/normalization/batch_normalization.py:142: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.5715 - f1_score: 0.3444 - loss: 0.2390
Epoch 2/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7026 - f1_score: 0.4932 - loss: 0.2179
Epoch 3/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7647 - f1_score: 0.6108 - loss: 0.1824
Epoch 4/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7737 - f1_score: 0.6473 - loss: 0.1615
Epoch 5/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7794 - f1_score: 0.6550 - loss: 0.1497
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7879 - f1_score: 0.6782 - loss: 0.5400


In [ ]:
m6_eval

{'loss': 0.5925179123878479,
 'accuracy': 0.7649999856948853,
 'f1_score': 0.6438450217247009}

Sprawdzenie poprawności wyniku

In [ ]:
assert 0.65 <= m6_eval['accuracy']
assert 0.6 <= m6_eval['f1_score']

In [79]:
y_pred_proba_m6 = model6.predict(X_test)
yhat_model6 = np.argmax(y_pred_proba_m6, axis=1)
y_test_labels = np.argmax(y_test, axis=1)

print(classification_report(y_test_labels, yhat_model6, zero_division=0))

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
              precision    recall  f1-score   support

           0       0.89      0.84      0.86       694
           1       0.77      0.70      0.73       215
           2       0.31      0.49      0.38        91

    accuracy                           0.78      1000
   macro avg       0.66      0.68      0.66      1000
weighted avg       0.81      0.78      0.79      1000



<div class='alert alert-block alert-info'>
 Otrzymane wyniki nie wyglądają na istotnie lepsze/gorsze od tych, otrzymywanych podczas równoważenia zbioru. Zastosowanie wag dla klas jest po prostu kolejnym narzędziem, po które warto sięgnąć w sytuacji, gdy mamy do czynienia z niezbalansowanymi klasami w zbiorze uczącym.
</div>

# Dalsze eksperymenty

Jeśli chcesz, przepowadź dalesze eksperymenty na przedstawionym zbiorze danych, obejmujące np. poszukiwanie odpowiedniej architektury sieci oraz hiperparametrów. Spróbuj zastosować różne metody równoważenia zbiorów, z odmiennymi parametrami. Może uda Ci się uzyskać zadowalajace rezultaty?